# FaceDet_Train — 얼굴 검출기 직접 학습 (WIDER FACE, YOLO11n, 1클래스)

OpenCV YuNet(`face_detection_yunet_2023mar.onnx`) 대신 쓸 **우리 얼굴 검출기**를 만듭니다. 가위바위보(RPS) 때와 같은 방식: ultralytics YOLO11n → tflite INT8.
절차 설명: `FaceReading_3/FaceDet_얼굴검출_학습_안내.html`

**만드는 것 (보드로 복사)**: `save/face_det_int8.tflite` (+ `facedet_results.png`, `facedet_samples.png`)

**데이터 (둘 중 하나)**
- A. 자동: 이 노트북이 torchvision 으로 WIDER FACE 를 내려받음 (약 1.8GB, 드라이브 업로드 불필요)
- B. 수동: 공식 페이지/Kaggle 에서 `WIDER_train.zip`, `WIDER_val.zip`, `wider_face_split.zip` 을 받아 드라이브 `files/WIDER/` 에 올려 두면 그걸 씀

런타임 **GPU(T4)**. 변환 10분 + 학습 60~90분.


## 1. 드라이브 · 경로

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive'); colab = True
except Exception:
    colab = False

In [ ]:
import os, sys, glob, time, zipfile, shutil, random
if colab:
    BASE = '/content/drive/MyDrive/ai_semiconductor/Python/examples/'
    DATA_DIR = BASE + 'files/WIDER/'; save_dir = BASE + 'files/save/'; work_dir = '/content/wider'
else:
    DATA_DIR = '../files/WIDER/'; save_dir = '../files/save/'; work_dir = './wider'
os.makedirs(work_dir, exist_ok=True); os.makedirs(save_dir, exist_ok=True)

## 2. 데이터 준비 (드라이브에 zip 이 있으면 그걸, 없으면 torchvision 으로 자동 다운로드)

In [ ]:
zips = {n: DATA_DIR + n for n in ('WIDER_train.zip', 'WIDER_val.zip', 'wider_face_split.zip')}
if all(os.path.exists(p) for p in zips.values()):
    print('드라이브 zip 사용'); 
    for p in zips.values():
        zipfile.ZipFile(p).extractall(work_dir)
    ROOT = work_dir
else:
    print('torchvision 으로 다운로드 (약 1.8GB) …')
    import torchvision
    torchvision.datasets.WIDERFace(work_dir, split='train', download=True)
    torchvision.datasets.WIDERFace(work_dir, split='val', download=True)
    ROOT = work_dir + '/widerface'
GT_TRAIN = glob.glob(ROOT + '/**/wider_face_train_bbx_gt.txt', recursive=True)[0]
GT_VAL   = glob.glob(ROOT + '/**/wider_face_val_bbx_gt.txt',   recursive=True)[0]
IMG_TRAIN = glob.glob(ROOT + '/**/WIDER_train/images', recursive=True)[0]
IMG_VAL   = glob.glob(ROOT + '/**/WIDER_val/images',   recursive=True)[0]
print(GT_TRAIN); print(IMG_TRAIN)

## 3. WIDER 라벨 → YOLO 라벨 변환
WIDER 형식: 파일명 / 얼굴 수 / 한 줄에 `x y w h blur expression illumination invalid occlusion pose`.
키오스크용이므로 **너무 작은 얼굴(짧은 변 20px 미만), invalid, 심한 가림(occlusion=2)** 은 빼고, 얼굴이 20개 넘는 군중 사진도 뺍니다.

In [ ]:
import cv2
YOLO_DIR = work_dir + '/yolo'
for s in ('train', 'val'):
    os.makedirs(f'{YOLO_DIR}/images/{s}', exist_ok=True); os.makedirs(f'{YOLO_DIR}/labels/{s}', exist_ok=True)

MIN_SIDE, MAX_FACES = 20, 20
N_TRAIN, N_VAL = 8000, 1000          # 사용할 사진 수 (전체 12,880 / 3,226). 시간 여유가 있으면 늘리기

def parse(gt_path):
    lines = open(gt_path).read().split('\n'); i = 0; out = {}
    while i < len(lines):
        name = lines[i].strip(); i += 1
        if not name: continue
        n = int(lines[i].strip()); i += 1
        boxes = []
        for _ in range(max(n, 1)):                        # n==0 이어도 더미 줄 하나가 있음
            t = lines[i].split(); i += 1
            if n == 0: continue
            x, y, w, h = map(int, t[:4]); invalid, occl = int(t[7]), int(t[8])
            if invalid or occl == 2 or min(w, h) < MIN_SIDE: continue
            boxes.append((x, y, w, h))
        if boxes and len(boxes) <= MAX_FACES: out[name] = boxes
    return out

def export(gt, img_root, split, limit):
    names = sorted(gt); random.seed(0); random.shuffle(names); names = names[:limit]; kept = 0
    for name in names:
        src = os.path.join(img_root, name); img = cv2.imread(src)
        if img is None: continue
        H, W = img.shape[:2]; base = name.replace('/', '__')
        shutil.copy(src, f'{YOLO_DIR}/images/{split}/{base}')
        with open(f'{YOLO_DIR}/labels/{split}/{os.path.splitext(base)[0]}.txt', 'w') as f:
            for x, y, w, h in gt[name]:
                f.write('0 %.6f %.6f %.6f %.6f\n' % ((x + w / 2) / W, (y + h / 2) / H, w / W, h / H))
        kept += 1
    print(split, kept, '장')

export(parse(GT_TRAIN), IMG_TRAIN, 'train', N_TRAIN)
export(parse(GT_VAL),   IMG_VAL,   'val',   N_VAL)
open(f'{YOLO_DIR}/data.yaml', 'w').write(f'path: {YOLO_DIR}\ntrain: images/train\nval: images/val\nnc: 1\nnames: [face]\n')
print(open(f'{YOLO_DIR}/data.yaml').read())

In [ ]:
import matplotlib.pyplot as plt
files = sorted(glob.glob(f'{YOLO_DIR}/images/train/*'))[:6]
plt.figure(figsize=(14, 3))
for k, f in enumerate(files):
    img = cv2.imread(f); H, W = img.shape[:2]
    for line in open(f.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'):
        _, cx, cy, w, h = map(float, line.split())
        cv2.rectangle(img, (int((cx - w/2) * W), int((cy - h/2) * H)), (int((cx + w/2) * W), int((cy + h/2) * H)), (0, 255, 0), 2)
    plt.subplot(1, 6, k + 1); plt.imshow(img[:, :, ::-1]); plt.axis('off')
plt.suptitle('YOLO 라벨 확인 (초록 박스가 얼굴에 있어야 함)'); plt.show()

## 4. YOLO11n 학습 (RPS 와 같은 방식)

In [ ]:
!pip -q install ultralytics
from ultralytics import YOLO
IMGSZ = 320                                   # 보드 속도 때문에 320. 정확도를 더 원하면 416
model = YOLO('yolo11n.pt')
res = model.train(data=f'{YOLO_DIR}/data.yaml', epochs=30, imgsz=IMGSZ, batch=64, project=work_dir + '/runs', name='face', exist_ok=True,
                  patience=10, cos_lr=True)

## 5. 평가 · 그림

In [ ]:
best = YOLO(work_dir + '/runs/face/weights/best.pt')
m = best.val(data=f'{YOLO_DIR}/data.yaml', imgsz=IMGSZ)
print('mAP50 = %.3f   mAP50-95 = %.3f' % (m.box.map50, m.box.map))
shutil.copy(work_dir + '/runs/face/results.png', save_dir + 'facedet_results.png')      # 학습 곡선
vals = sorted(glob.glob(f'{YOLO_DIR}/images/val/*'))[:6]
plt.figure(figsize=(14, 3))
for k, f in enumerate(vals):
    r = best.predict(f, imgsz=IMGSZ, conf=0.4, verbose=False)[0]
    plt.subplot(1, 6, k + 1); plt.imshow(r.plot()[:, :, ::-1]); plt.axis('off')
plt.suptitle('our face detector (val)'); plt.savefig(save_dir + 'facedet_samples.png', dpi=130, bbox_inches='tight'); plt.show()

## 6. TFLite INT8 변환 · 저장 (RPS 의 best_int8.tflite 와 같은 형식)

In [ ]:
out = best.export(format='tflite', imgsz=IMGSZ, int8=True, data=f'{YOLO_DIR}/data.yaml')
cands = glob.glob(work_dir + '/runs/face/weights/*_saved_model/*int8*.tflite') or glob.glob(work_dir + '/runs/face/weights/**/*.tflite', recursive=True)
print(cands)
src = [c for c in cands if 'int8' in c and 'full' not in c][0] if any('int8' in c for c in cands) else cands[0]
shutil.copy(src, save_dir + 'face_det_int8.tflite')
print('face_det_int8.tflite  %.1f MB' % (os.path.getsize(save_dir + 'face_det_int8.tflite') / 1e6))

# 출력 형식 확인: (1, 5, N) = [cx, cy, w, h, face_score] × N
import tensorflow as tf, numpy as np
it = tf.lite.Interpreter(model_path=save_dir + 'face_det_int8.tflite'); it.allocate_tensors()
i_, o_ = it.get_input_details()[0], it.get_output_details()[0]
print('input', i_['shape'], i_['dtype'], '| output', o_['shape'], o_['dtype'])

## 7. 보드 적용
`save/face_det_int8.tflite` 를 보드 폴더에 넣으면 `face_fortune.py` 가 자동으로 YuNet 대신 이 모델을 씁니다
(파일이 없으면 YuNet). 켤 때 `얼굴 검출: our YOLO (face_det_int8.tflite)` 가 찍히면 적용된 것.